# Research — EDA & findings: next-direction prediction

**Goal.** Honestly answer: *is there a learnable, cost-surviving short-term directional signal in the engine's features?*

**Honesty up front.** The live `engine.db` is currently **not trainable** (too few rows, no directional variance). This notebook therefore runs on the **disclosed synthetic session** (`synthetic.py`, momentum-persistence φ=0.15 + costs) to demonstrate the methodology. It is *not* a performance claim. The same code runs on live data via `--source db` once enough is collected.

Run from the repo root with the research venv kernel (`agent/research/.venv`).

In [ ]:
import json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

from agent.research import features as F
from agent.research.synthetic import load_synthetic

# Show that live data is not trainable, then proceed on synthetic.
try:
    df_db = F.build_matrix(*F.load_sqlite('data/engine.db'))
    print('live db:', F.trainability(df_db))
except Exception as e:
    print('live db load error:', e)

df = F.build_matrix(*load_synthetic())
print('synthetic:', F.trainability(df))
df.shape

## 1. Class balance & label
Forward-60s direction. A near-50/50 split is expected for FX — anything wildly off would hint at leakage.

In [ ]:
print(df['y'].value_counts(normalize=True).round(3).to_dict())
df.groupby('symbol')['y'].mean().round(3)

## 2. Feature distributions & relationship to the label
Point-biserial-ish: mean of each feature by outcome class. Big gaps = potentially informative.

In [ ]:
cols = ['mom_ret','mom_sign','vol_score','confidence','trade_quality','roll_mean_5','roll_std_5','roll_ret_sum_3']
by_class = df.groupby('y')[cols].mean().T
by_class['gap'] = (by_class[1] - by_class[0])
by_class.sort_values('gap', key=abs, ascending=False).round(5)

In [ ]:
# Correlation of each feature with the (signed) forward return — the honest target.
df[cols].corrwith(df['fwd_ret']).sort_values(key=abs, ascending=False).round(4)

## 3. Model result (walk-forward Logistic Regression)
Produced by `python -m agent.research.train`. We load the saved metrics rather than re-train inline.

In [ ]:
r = json.load(open('agent/research/results/model_results.json'))
m = r['metrics']
print(f"AUC {m['auc']:.3f}  |  permutation null {m['permutation_auc_mean']:.3f} ± {m['permutation_auc_std']:.3f}  |  p={m['p_value']}")
print(f"precision {m['precision']:.2f}  recall {m['recall']:.2f}  Brier {m['brier']:.3f}")
pd.DataFrame(r['net_of_cost']['threshold_curve'])[['margin','n_trades','gross_pct','net_pct','hit_rate']].round(2)

In [ ]:
pd.DataFrame(r['feature_importance']).head(8)[['feature','coef']]

## 4. Findings (honest)

1. **A weak directional signal exists and is statistically real** — OOS AUC ≈ 0.545 vs a permutation null of ≈ 0.498 (p ≈ 0.03). `mom_sign` (momentum persistence) is the dominant feature, exactly the planted effect.
2. **It does not survive costs.** Gross ≈ +8.5% becomes net ≈ −57% over ~3,900 trades; the loss shrinks but stays negative at every selectivity threshold. **No deployable edge.**
3. **Calibration is decent** (Brier ≈ 0.25, reliability near the diagonal) — the probabilities are honest, even though the edge is uneconomic.
4. **Implication.** At 30s cadence on a momentum-persistence signal, transaction costs dominate. A real edge must come from (a) lower turnover, (b) a stronger/longer-horizon signal, or (c) cheaper execution — *and must be proven net of cost.* This is the difference between a quant and a backtest tourist.
5. **Next:** collect weeks of live session-aligned data, re-run with `--source db`, and re-evaluate. The pipeline is ready; the data is not yet.